In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
print("Libraries loaded")

Libraries loaded


In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()
    DATA_PATH = list(uploaded.keys())[0]
except ImportError:
    DATA_PATH = "energy_efficiency_data.csv"
print("Data file:", DATA_PATH)

Saving energy_efficiency_data (2).csv to energy_efficiency_data (2).csv
Data file: energy_efficiency_data (2).csv


In [ ]:
OUT_DIR = "eda_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

FEATURES = [
    "Relative_Compactness", "Surface_Area", "Wall_Area", "Roof_Area",
    "Overall_Height", "Orientation", "Glazing_Area", "Glazing_Area_Distribution",
]
TARGETS = ["Heating_Load", "Cooling_Load"]   # Y1, Y2

def save_fig(name):
    path = os.path.join(OUT_DIR, name)
    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close()
    print(f"saved -> {path}")

In [ ]:
df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
df.head()

Shape: (768, 10)


,Relative_Compactness,Surface_Area,Wall_Area,Roof_Area,Overall_Height,Orientation,Glazing_Area,Glazing_Area_Distribution,Heating_Load,Cooling_Load
0,0.98,514.5,294.0,110.25,7.0,2,0.0,0,15.55,21.33
1,0.98,514.5,294.0,110.25,7.0,3,0.0,0,15.55,21.33
2,0.98,514.5,294.0,110.25,7.0,4,0.0,0,15.55,21.33
3,0.98,514.5,294.0,110.25,7.0,5,0.0,0,15.55,21.33
4,0.90,563.5,318.5,122.50,7.0,2,0.0,0,20.84,28.28


2️⃣ Preprocessing & Feature Engineering

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
import joblib

CAT_COLS = ["Orientation", "Glazing_Area_Distribution"]

X = df[FEATURES]
y = df[TARGETS]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print("Train:", X_train.shape, "| Test:", X_test.shape)

Train: (614, 8) | Test: (154, 8)


In [ ]:
def add_features(X, engineer=True, drop=()):
    X = X.copy()
    if engineer:
        X["Wall_Surface_Ratio"] = X["Wall_Area"] / X["Surface_Area"]
        X["Glazing_x_Height"] = X["Glazing_Area"] * X["Overall_Height"]
    return X.drop(columns=list(drop))

def build_preprocessor(scale=True, onehot=True, engineer=True, drop=()):
    num_cols = [c for c in FEATURES if c not in CAT_COLS and c not in drop]
    if engineer:
        num_cols += ["Wall_Surface_Ratio", "Glazing_x_Height"]

    num_tf = StandardScaler() if scale else "passthrough"
    cat_tf = OneHotEncoder(handle_unknown="ignore", sparse_output=False) if onehot else "passthrough"

    ct = ColumnTransformer([
        ("num", num_tf, num_cols),
        ("cat", cat_tf, CAT_COLS),
    ])
    return Pipeline([
        ("features", FunctionTransformer(add_features,
                                         kw_args={"engineer": engineer, "drop": drop})),
        ("transform", ct),
    ])

# نسخة للموديلات الخطية (scaling + onehot) ونسخة للـ trees (من غير scaling)
prep_linear = build_preprocessor(scale=True, onehot=True)
prep_tree   = build_preprocessor(scale=False, onehot=False)

Xtr_lin = prep_linear.fit_transform(X_train)   # fit على الـ train بس
Xte_lin = prep_linear.transform(X_test)
print("Linear-ready shape:", Xtr_lin.shape)

Linear-ready shape: (614, 18)


In [ ]:
experiments = {
    "Baseline (original features)":      dict(engineer=False),
    "+ New features":                    dict(engineer=True),
    "Drop Surface_Area":                 dict(engineer=False, drop=("Surface_Area",)),
    "Drop Surface_Area + Roof_Area":     dict(engineer=False, drop=("Surface_Area", "Roof_Area")),
    "New features + drop Surface_Area":  dict(engineer=True,  drop=("Surface_Area",)),
}

rows = []
for name, cfg in experiments.items():
    pipe = Pipeline([
        ("prep", build_preprocessor(scale=True, onehot=True, **cfg)),
        ("model", LinearRegression()),
    ])
    r2 = cross_val_score(pipe, X_train, y_train, cv=5, scoring="r2")
    rmse = -cross_val_score(pipe, X_train, y_train, cv=5,
                            scoring="neg_root_mean_squared_error")
    rows.append({"Experiment": name, "CV R²": r2.mean().round(4), "CV RMSE": rmse.mean().round(3)})

exp_df = pd.DataFrame(rows).sort_values("CV R²", ascending=False)
exp_df.to_csv(os.path.join(OUT_DIR, "feature_experiments.csv"), index=False)
exp_df

,Experiment,CV R²,CV RMSE
1,+ New features,0.9083,2.921
4,New features + drop Surface_Area,0.9083,2.921
0,Baseline (original features),0.9019,3.029
2,Drop Surface_Area,0.9019,3.029
3,Drop Surface_Area + Roof_Area,0.8987,3.077


In [ ]:
os.makedirs("processed", exist_ok=True)

X_train.assign(**y_train).to_csv("processed/train.csv", index=False)
X_test.assign(**y_test).to_csv("processed/test.csv", index=False)
joblib.dump(prep_linear, "processed/preprocessor_linear.joblib")
joblib.dump(prep_tree,   "processed/preprocessor_tree.joblib")
print("saved")

saved


In [ ]:
import os
os.makedirs("src", exist_ok=True)
open("src/__init__.py", "w").close()

code = '''
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

FEATURES = [
    "Relative_Compactness", "Surface_Area", "Wall_Area", "Roof_Area",
    "Overall_Height", "Orientation", "Glazing_Area", "Glazing_Area_Distribution",
]
TARGETS = ["Heating_Load", "Cooling_Load"]
CAT_COLS = ["Orientation", "Glazing_Area_Distribution"]

def add_features(X, engineer=True, drop=()):
    X = X.copy()
    if engineer:
        X["Wall_Surface_Ratio"] = X["Wall_Area"] / X["Surface_Area"]
        X["Glazing_x_Height"] = X["Glazing_Area"] * X["Overall_Height"]
    return X.drop(columns=list(drop))

def build_preprocessor(scale=True, onehot=True, engineer=True, drop=()):
    num_cols = [c for c in FEATURES if c not in CAT_COLS and c not in drop]
    if engineer:
        num_cols += ["Wall_Surface_Ratio", "Glazing_x_Height"]
    num_tf = StandardScaler() if scale else "passthrough"
    cat_tf = OneHotEncoder(handle_unknown="ignore", sparse_output=False) if onehot else "passthrough"
    ct = ColumnTransformer([("num", num_tf, num_cols), ("cat", cat_tf, CAT_COLS)])
    return Pipeline([
        ("features", FunctionTransformer(add_features, kw_args={"engineer": engineer, "drop": drop})),
        ("transform", ct),
    ])
'''
with open("src/preprocessing.py", "w") as f:
    f.write(code)

print(os.listdir("src"))

['preprocessing.py', '__init__.py']


In [ ]:
import joblib
from src.preprocessing import build_preprocessor, FEATURES, TARGETS

prep_linear = build_preprocessor(scale=True, onehot=True)
prep_tree   = build_preprocessor(scale=False, onehot=False)
prep_linear.fit(X_train)
prep_tree.fit(X_train)

os.makedirs("processed", exist_ok=True)
X_train.assign(**y_train).to_csv("processed/train.csv", index=False)
X_test.assign(**y_test).to_csv("processed/test.csv", index=False)
joblib.dump(prep_linear, "processed/preprocessor_linear.joblib")
joblib.dump(prep_tree,   "processed/preprocessor_tree.joblib")

print(joblib.load("processed/preprocessor_linear.joblib").transform(X_test).shape)

(154, 18)
